# FINAL FULL-FLEDGED BACKEND — COMPLETE END-TO-END VERIFICATION, REPAIR & HARDENING

This notebook verifies the backend step by step following the provided phase outline. It is designed to confirm configuration, MongoDB Atlas connectivity, FastAPI startup, device APIs, IoT data ingestion, feature extraction, ML/detection services, notifications, dashboard APIs, MQTT integration, and pytest validation.


## 1. Import Required Libraries

Import OS, dotenv, HTTP client, requests, pymongo, FastAPI test client, and ML/drift detection libraries needed for verification.


In [ ]:
import os
import json
from pathlib import Path

from dotenv import load_dotenv
import requests
from pymongo import MongoClient
from fastapi.testclient import TestClient

# Optional: import project modules when available
project_import_errors = []
try:
    from app.config import settings
    from app.main import app
    from app.database import connect_db, close_db, ping_db
    from app.services.feature_service import extract_features
    from app.services.anomaly_service import analyze_anomaly
    from app.services.drift_service import update_drift
    from app.services.trust_service import calculate_trust_score
except Exception as exc:
    project_import_errors.append((type(exc).__name__, str(exc)))

project_import_errors


In [ ]:

# 2. Load .env and Configuration
load_dotenv(dotenv_path=Path('c:/IOT_Project/backend/.env'))

env_loaded = {
    'MONGODB_URI_present': 'MONGODB_URI' in os.environ and bool(os.environ['MONGODB_URI']),
    'DATABASE_NAME_present': 'DATABASE_NAME' in os.environ and bool(os.environ['DATABASE_NAME']),
    'MQTT_BROKER_present': 'MQTT_BROKER' in os.environ and bool(os.environ['MQTT_BROKER']),
    'MQTT_PORT_present': 'MQTT_PORT' in os.environ and bool(os.environ['MQTT_PORT']),
}

config_loaded = False
config_errors = []
mongodb_uri_present = False
try:
    config_loaded = True
    mongodb_uri_present = bool(settings.MONGODB_URI.get_secret_value())
except Exception as exc:
    config_errors.append((type(exc).__name__, str(exc)))

{
    'project_import_errors': project_import_errors,
    'env_loaded': env_loaded,
    'config_loaded': config_loaded,
    'mongodb_uri_present': mongodb_uri_present,
    'config_errors': config_errors,
}


In [1]:
import subprocess

print('--- GIT STATUS ---')
status = subprocess.run(['git', 'status', '--short'], cwd='c:/IOT_Project/backend', capture_output=True, text=True)
print(status.stdout)
print('--- GIT DIFF app/config.py app/database.py ---')
diff = subprocess.run(['git', 'diff', '--', 'app/config.py', 'app/database.py'], cwd='c:/IOT_Project/backend', capture_output=True, text=True)
print(diff.stdout[:2000])

print('\n--- DIAGNOSTIC ---')
from pathlib import Path
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path=Path('c:/IOT_Project/backend/.env'))
print('MONGODB_URI_in_env', 'MONGODB_URI' in os.environ and bool(os.environ['MONGODB_URI']))
print('DATABASE_NAME_in_env', 'DATABASE_NAME' in os.environ and bool(os.environ['DATABASE_NAME']))
print('MQTT_BROKER_in_env', 'MQTT_BROKER' in os.environ and bool(os.environ['MQTT_BROKER']))
print('MQTT_PORT_in_env', 'MQTT_PORT' in os.environ and bool(os.environ['MQTT_PORT']))

from app.config import settings
print('CONFIG_MONGODB_URI_present', bool(settings.MONGODB_URI.get_secret_value()))
print('CONFIG_DATABASE_NAME', settings.DATABASE_NAME)

from pymongo import MongoClient
try:
    client = MongoClient(settings.mongodb_uri, serverSelectionTimeoutMS=5000)
    client.admin.command('ping')
    print('PING', 'PASS')
    db = client[settings.DATABASE_NAME]
    print('DB_EXISTS', settings.DATABASE_NAME in client.list_database_names())
    names = db.list_collection_names()
    print('COLLECTIONS', {name: name in names for name in ['devices', 'device_analysis', 'notifications']})
except Exception as exc:
    print('MONGO_ERROR', type(exc).__name__, str(exc))
finally:
    try:
        client.close()
    except Exception:
        pass


--- GIT STATUS ---

--- GIT DIFF app/config.py app/database.py ---
diff --git a/app/config.py b/app/database.py
index 671b0fd..eb712df 100644
--- a/app/config.py
+++ b/app/database.py
@@ -1,51 +1,240 @@
-from typing import Optional
-from urllib.parse import quote_plus
+from typing import Optional, List, Any
+from datetime import datetime, timedelta
 
-from pydantic import Field, SecretStr
-from pydantic_settings import BaseSettings, SettingsConfigDict
+from motor.motor_asyncio import AsyncIOMotorClient
 
 
-def _sanitize_mongodb_uri(uri: str) -> str:
-    """Normalize MongoDB URI credentials if the password contains reserved chars."""
-    if not uri:
-        return uri
-    if not (uri.startswith('mongodb://') or uri.startswith('mongodb+srv://')):
-        return uri
+class DatabaseConnectionError(RuntimeError):
+    pass
 
-    scheme, rest = uri.split('://', 1)
-    if '@' not in rest:
-        return uri
+from app.config import settings
 
-    # Preserve the rightmost @ as the sep

ModuleNotFoundError: No module named 'pydantic_settings'

In [1]:
import subprocess
from pathlib import Path
from dotenv import load_dotenv
import os

# Check repository file modifications
cwd = Path('c:/IOT_Project/backend')
status = subprocess.run(['git', 'status', '--short'], cwd=cwd, capture_output=True, text=True)
diff_config = subprocess.run(['git', 'diff', 'HEAD', '--', 'app/config.py'], cwd=cwd, capture_output=True, text=True)
diff_database = subprocess.run(['git', 'diff', 'HEAD', '--', 'app/database.py'], cwd=cwd, capture_output=True, text=True)

print('GIT_STATUS:')
print(status.stdout or '<clean>')
print('\nGIT_DIFF app/config.py:')
print(diff_config.stdout[:4000] or '<no diff>')
print('\nGIT_DIFF app/database.py:')
print(diff_database.stdout[:4000] or '<no diff>')

# Load .env and config
load_dotenv(dotenv_path=cwd / '.env')
print('\nENV VARS:')
print('MONGODB_URI' in os.environ and bool(os.environ['MONGODB_URI']))
print('DATABASE_NAME' in os.environ and bool(os.environ['DATABASE_NAME']))
print('MQTT_BROKER' in os.environ and bool(os.environ['MQTT_BROKER']))
print('MQTT_PORT' in os.environ and bool(os.environ['MQTT_PORT']))

from app.config import settings
print('\nCONFIG MONGODB_URI present:', bool(settings.MONGODB_URI.get_secret_value()))
print('CONFIG DATABASE_NAME:', settings.DATABASE_NAME)

from pymongo import MongoClient
try:
    client = MongoClient(settings.mongodb_uri, serverSelectionTimeoutMS=5000)
    client.admin.command('ping')
    print('\nMONGO PING PASS')
    db_exists = settings.DATABASE_NAME in client.list_database_names()
    print('DB_EXISTS:', db_exists)
    db = client[settings.DATABASE_NAME]
    names = db.list_collection_names()
    print('COLLECTIONS:', {name: name in names for name in ['devices', 'device_analysis', 'notifications']})
except Exception as exc:
    print('\nMONGO_ERROR:', type(exc).__name__, str(exc))
finally:
    try:
        client.close()
    except Exception:
        pass


GIT_STATUS:
<clean>

GIT_DIFF app/config.py:
<no diff>

GIT_DIFF app/database.py:
<no diff>

ENV VARS:
True
False
True
True

CONFIG MONGODB_URI present: True
CONFIG DATABASE_NAME: iot_trust_db

MONGO PING PASS
DB_EXISTS: False
COLLECTIONS: {'devices': False, 'device_analysis': False, 'notifications': False}


In [1]:
import os
import sys
import time
import json
import subprocess
from pathlib import Path
from urllib.parse import urlparse

import requests
from dotenv import load_dotenv
from pymongo import MongoClient

# Ensure current working directory is the backend root for uvicorn and config loading.
backend_root = Path('c:/IOT_Project/backend')
load_dotenv(dotenv_path=backend_root / '.env')

# Start uvicorn using the notebook's Python environment.
uvicorn_cmd = [sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '127.0.0.1', '--port', '8000', '--log-level', 'warning']
server_proc = subprocess.Popen(uvicorn_cmd, cwd=backend_root, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

try:
    base_url = 'http://127.0.0.1:8000'
    start_time = time.time()
    server_ready = False
    while time.time() - start_time < 30:
        try:
            resp = requests.get(f'{base_url}/health', timeout=3)
            if resp.status_code == 200:
                server_ready = True
                break
        except requests.RequestException:
            pass
        time.sleep(1)

    print('server_ready', server_ready)
    if not server_ready:
        out, err = server_proc.communicate(timeout=5)
        print('SERVER_STARTUP_FAILED')
        print('stdout:', out)
        print('stderr:', err)
        raise RuntimeError('FastAPI did not start in time')

    def api_get(path):
        return requests.get(base_url + path, timeout=10)

    def api_post(path, payload):
        return requests.post(base_url + path, json=payload, timeout=10)

    # Step 2: verify /health and /docs
    health_resp = api_get('/health')
    docs_resp = api_get('/docs')
    print('health_status', health_resp.status_code)
    print('docs_status', docs_resp.status_code)

    # Step 3-7: device creation and retrieval
    device_payload = {
        'device_id': 'DEMO001',
        'device_name': 'Smart Camera Demo',
        'device_type': 'smart_camera',
    }
    register_resp = api_post('/api/devices', device_payload)
    print('register_status', register_resp.status_code)
    print('register_body', register_resp.json())

    list_resp = api_get('/api/devices')
    print('list_status', list_resp.status_code)
    print('list_count', len(list_resp.json()) if list_resp.status_code == 200 else None)

    get_resp = api_get('/api/devices/DEMO001')
    print('get_status', get_resp.status_code)
    print('get_body', get_resp.json() if get_resp.status_code == 200 else get_resp.text)

    # Step 8-10: analysis endpoint and MongoDB storage
    analysis_payload = {
        'packet_rate': 500,
        'behavior_score': 90,
        'network_score': 92,
        'firmware_score': 95,
    }
    analyze_resp = api_post('/api/devices/DEMO001/analyze', analysis_payload)
    print('analyze_status', analyze_resp.status_code)
    print('analyze_body', analyze_resp.json() if analyze_resp.status_code == 200 else analyze_resp.text)

    # Query MongoDB directly to verify stored data and collections
    from app.config import settings
    client = MongoClient(settings.mongodb_uri, serverSelectionTimeoutMS=5000)
    client.admin.command('ping')
    db = client[settings.DATABASE_NAME]
    collections = set(db.list_collection_names())
    print('mongo_collections', collections)
    device_doc = db['devices'].find_one({'device_id': 'DEMO001'})
    print('device_doc_exists', device_doc is not None)
    analysis_doc = db['device_analysis'].find_one({'device_id': 'DEMO001'})
    print('analysis_doc_exists', analysis_doc is not None)
    notification_doc = db['notifications'].find_one({'device_id': 'DEMO001'})
    print('notification_doc_exists', notification_doc is not None)

    # Dashboard endpoints
    summary_resp = api_get('/api/dashboard/summary')
    dashboard_devices_resp = api_get('/api/dashboard/devices')
    history_resp = api_get('/api/dashboard/devices/DEMO001/history')
    print('dashboard_summary', summary_resp.status_code, summary_resp.json() if summary_resp.status_code == 200 else summary_resp.text)
    print('dashboard_devices', dashboard_devices_resp.status_code, dashboard_devices_resp.json() if dashboard_devices_resp.status_code == 200 else dashboard_devices_resp.text)
    print('dashboard_history', history_resp.status_code, history_resp.json() if history_resp.status_code == 200 else history_resp.text)

    # Notification endpoints
    notifications_resp = api_get('/api/notifications')
    notifications_for_device_resp = api_get('/api/notifications/DEMO001')
    print('notifications_list', notifications_resp.status_code, len(notifications_resp.json()) if notifications_resp.status_code == 200 else None)
    print('notifications_device', notifications_for_device_resp.status_code, notifications_for_device_resp.json() if notifications_for_device_resp.status_code == 200 else notifications_for_device_resp.text)

    # Pytest
    pytest_proc = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=backend_root, capture_output=True, text=True)
    print('pytest_returncode', pytest_proc.returncode)
    print(pytest_proc.stdout)
    print(pytest_proc.stderr)
finally:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=10)
    except Exception:
        server_proc.kill()


server_ready False


TimeoutExpired: Command '['c:\\IOT_Project\\.venv\\Scripts\\python.exe', '-m', 'uvicorn', 'app.main:app', '--host', '127.0.0.1', '--port', '8000', '--log-level', 'warning']' timed out after 5 seconds

In [2]:
import sys
import subprocess
print('sys.executable:', sys.executable)
print('python version:', sys.version)
print('pydantic-settings installed:', subprocess.run([sys.executable, '-m', 'pip', 'show', 'pydantic-settings'], capture_output=True, text=True).stdout.strip())
print('git status:', subprocess.run(['git', 'status', '--short'], cwd='c:/IOT_Project/backend', capture_output=True, text=True).stdout)
print('git diff config:', subprocess.run(['git', 'diff', 'HEAD', '--', 'app/config.py'], cwd='c:/IOT_Project/backend', capture_output=True, text=True).stdout[:2000])
print('git diff database:', subprocess.run(['git', 'diff', 'HEAD', '--', 'app/database.py'], cwd='c:/IOT_Project/backend', capture_output=True, text=True).stdout[:2000])


sys.executable: c:\IOT_Project\.venv\Scripts\python.exe
python version: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
pydantic-settings installed: 
git status: 
git diff config: 
git diff database: 


In [ ]:
print('notebook execution test')


In [3]:
import os
import sys
import time
import subprocess
from pathlib import Path

import requests
from dotenv import load_dotenv
from pymongo import MongoClient

# Setup environment and backend path
backend_root = Path('c:/IOT_Project/backend')
load_dotenv(dotenv_path=backend_root / '.env')

uvicorn_cmd = [sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '127.0.0.1', '--port', '8000', '--log-level', 'info']
print('Starting server with:', uvicorn_cmd)
server_proc = subprocess.Popen(
    uvicorn_cmd,
    cwd=backend_root,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

try:
    start_time = time.time()
    server_ready = False
    stderr_lines = []
    while time.time() - start_time < 30:
        line = server_proc.stderr.readline()
        if line:
            stderr_lines.append(line)
            print(line, end='')
            if 'Application startup complete' in line or 'Uvicorn running on' in line:
                server_ready = True
                break
        else:
            time.sleep(0.2)

    if not server_ready:
        print('Server did not become ready in 30s')
        print('Captured stderr:')
        print(''.join(stderr_lines[-50:]))
        raise RuntimeError('FastAPI startup failed')

    base_url = 'http://127.0.0.1:8000'
    print('Checking /health and /docs')
    health_resp = requests.get(base_url + '/health', timeout=10)
    docs_resp = requests.get(base_url + '/docs', timeout=10)
    print('health', health_resp.status_code)
    print('docs', docs_resp.status_code)

    device_payload = {
        'device_id': 'DEMO001',
        'device_name': 'Smart Camera Demo',
        'device_type': 'smart_camera',
    }
    register_resp = requests.post(base_url + '/api/devices', json=device_payload, timeout=10)
    print('register', register_resp.status_code, register_resp.text)

    list_resp = requests.get(base_url + '/api/devices', timeout=10)
    print('list', list_resp.status_code, list_resp.text)

    get_resp = requests.get(base_url + '/api/devices/DEMO001', timeout=10)
    print('get', get_resp.status_code, get_resp.text)

    analysis_payload = {
        'packet_rate': 500,
        'behavior_score': 90,
        'network_score': 92,
        'firmware_score': 95,
    }
    analyze_resp = requests.post(base_url + '/api/devices/DEMO001/analyze', json=analysis_payload, timeout=10)
    print('analyze', analyze_resp.status_code, analyze_resp.text)

    from app.config import settings
    client = MongoClient(settings.mongodb_uri, serverSelectionTimeoutMS=5000)
    client.admin.command('ping')
    db = client[settings.DATABASE_NAME]
    print('db exists', settings.DATABASE_NAME in client.list_database_names())
    names = db.list_collection_names()
    print('collections', names)
    device_doc = db['devices'].find_one({'device_id': 'DEMO001'})
    print('device exists', device_doc is not None)
    analysis_doc = db['device_analysis'].find_one({'device_id': 'DEMO001'})
    print('analysis exists', analysis_doc is not None)
    notification_doc = db['notifications'].find_one({'device_id': 'DEMO001'})
    print('notification exists', notification_doc is not None)

    print('dashboard summary', requests.get(base_url + '/api/dashboard/summary', timeout=10).status_code)
    print('dashboard devices', requests.get(base_url + '/api/dashboard/devices', timeout=10).status_code)
    print('dashboard history', requests.get(base_url + '/api/dashboard/devices/DEMO001/history', timeout=10).status_code)
    print('notifications list', requests.get(base_url + '/api/notifications', timeout=10).status_code)
    print('notifications device', requests.get(base_url + '/api/notifications/DEMO001', timeout=10).status_code)

    pytest_proc = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=backend_root, capture_output=True, text=True)
    print('pytest returncode', pytest_proc.returncode)
    print(pytest_proc.stdout)
    print(pytest_proc.stderr)
finally:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=10)
    except Exception:
        server_proc.kill()


Starting server with: ['c:\\IOT_Project\\.venv\\Scripts\\python.exe', '-m', 'uvicorn', 'app.main:app', '--host', '127.0.0.1', '--port', '8000', '--log-level', 'info']
INFO:     Started server process [24604]
INFO:     Waiting for application startup.
Could not connect to MQTT broker: [WinError 10061] No connection could be made because the target machine actively refused it
INFO:     Application startup complete.
Checking /health and /docs
health 200
docs 200
register 409 {"success":false,"message":"Device with this device_id already exists","device_id":"DEMO001"}
list 200 [{"device_id":"DEMO001","device_name":"Smart Camera Demo","device_type":"smart_camera","created_at":"2026-08-12T08:55:00.121000"}]
get 200 {"device_id":"DEMO001","device_name":"Smart Camera Demo","device_type":"smart_camera","created_at":"2026-08-12T08:55:00.121000"}
analyze 200 {"device_id":"DEMO001","input_features":{"packet_rate":500.0,"behavior_score":90.0,"network_score":92.0,"firmware_score":95.0},"anomaly_dete

In [4]:
import os
import sys
import subprocess
import socket
import time
from pathlib import Path

from dotenv import load_dotenv
from fastapi.testclient import TestClient

from app.config import settings
from app.main import app
from app.services.anomaly_service import get_default_isolation_forest, detect_anomaly
from app.services.drift_service import compute_behavior_metric
from pymongo import MongoClient

backend_root = Path('c:/IOT_Project/backend')
load_dotenv(dotenv_path=backend_root / '.env')

print('MQTT config:', settings.MQTT_BROKER, settings.MQTT_PORT, settings.MQTT_TOPIC)
if settings.MQTT_USERNAME and settings.MQTT_PASSWORD:
    print('MQTT auth configured')
else:
    print('MQTT auth not configured')

# Broker availability check
mqtt_available = False
broker_error = None
try:
    with socket.create_connection((settings.MQTT_BROKER, settings.MQTT_PORT), timeout=5):
        mqtt_available = True
except Exception as exc:
    broker_error = exc
print('mqtt_available', mqtt_available)
print('broker_error', type(broker_error).__name__, broker_error)

# Find an anomaly payload if possible.
model = get_default_isolation_forest()
test_candidates = [
    [500.0, 10.0, 10.0, 10.0],
    [300.0, 20.0, 20.0, 20.0],
    [200.0, 50.0, 10.0, 10.0],
    [100.0, 0.0, 0.0, 0.0],
    [1000.0, 30.0, 30.0, 30.0],
]
for candidate in test_candidates:
    result = detect_anomaly(candidate, model)
    print('candidate', candidate, 'prediction', result['prediction'], 'anomaly', result['anomaly_detected'], 'score', result['anomaly_score'])

# Use the first anomaly-producing payload
anomaly_payload = None
for candidate in test_candidates:
    if detect_anomaly(candidate, model)['anomaly_detected']:
        anomaly_payload = candidate
        break

print('selected_anomaly_payload', anomaly_payload)

client = TestClient(app)

with client:
    # Ensure the device exists
    reg = client.post('/api/devices', json={
        'device_id': 'DEMO001',
        'device_name': 'Smart Camera Demo',
        'device_type': 'smart_camera',
    })
    print('device_register_status', reg.status_code, reg.json())

    if anomaly_payload is None:
        raise RuntimeError('Could not find an anomaly payload from the candidate set')

    payload = {
        'packet_rate': anomaly_payload[0],
        'behavior_score': anomaly_payload[1],
        'network_score': anomaly_payload[2],
        'firmware_score': anomaly_payload[3],
    }
    analysis_resp = client.post('/api/devices/DEMO001/analyze', json=payload)
    print('analysis_status', analysis_resp.status_code)
    print('analysis_body', analysis_resp.json() if analysis_resp.status_code == 200 else analysis_resp.text)

    notifications_list = client.get('/api/notifications')
    notifications_device = client.get('/api/notifications/DEMO001')
    print('notifications_list_status', notifications_list.status_code, notifications_list.json())
    print('notifications_device_status', notifications_device.status_code)
    if notifications_device.status_code == 200:
        print('notifications_device_body', notifications_device.json())
    else:
        print('notifications_device_text', notifications_device.text)

# Verify MongoDB notification doc directly
mongo_client = MongoClient(settings.mongodb_uri, serverSelectionTimeoutMS=5000)
mongo_client.admin.command('ping')
db = mongo_client[settings.DATABASE_NAME]
notification_doc = db['notifications'].find_one({'device_id': 'DEMO001'})
print('mongo_notification_exists', notification_doc is not None)
if notification_doc:
    print('notification_doc_fields', {k: type(v).__name__ for k, v in notification_doc.items()})
    print('notification_doc_status', notification_doc.get('status'))

# Install pytest if missing and run the suite
pytest_installed = subprocess.run([sys.executable, '-m', 'pip', 'show', 'pytest'], capture_output=True, text=True)
print('pytest installed:', bool(pytest_installed.stdout.strip()))
if not pytest_installed.stdout.strip():
    print('Installing pytest...')
    install_proc = subprocess.run([sys.executable, '-m', 'pip', 'install', 'pytest'], capture_output=True, text=True)
    print(install_proc.stdout)
    print(install_proc.stderr)

pytest_proc_q = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=backend_root, capture_output=True, text=True)
print('pytest -q returncode', pytest_proc_q.returncode)
print(pytest_proc_q.stdout)
print(pytest_proc_q.stderr)

pytest_proc_v = subprocess.run([sys.executable, '-m', 'pytest', '-v'], cwd=backend_root, capture_output=True, text=True)
print('pytest -v returncode', pytest_proc_v.returncode)
print(pytest_proc_v.stdout)
print(pytest_proc_v.stderr)


RuntimeError: The starlette.testclient module requires the httpx2 package to be installed.
You can install this with:
    $ pip install httpx2


In [5]:
import os
import sys
import time
import socket
import subprocess
from pathlib import Path

import requests
from dotenv import load_dotenv
from pymongo import MongoClient

from app.config import settings
from app.services.anomaly_service import get_default_isolation_forest, detect_anomaly

backend_root = Path('c:/IOT_Project/backend')
load_dotenv(dotenv_path=backend_root / '.env')

print('MQTT host', settings.MQTT_BROKER)
print('MQTT port', settings.MQTT_PORT)
print('MQTT topic', settings.MQTT_TOPIC)
print('MQTT auth configured', bool(settings.MQTT_USERNAME and settings.MQTT_PASSWORD))

mqtt_available = False
mqtt_error = None
try:
    with socket.create_connection((settings.MQTT_BROKER, settings.MQTT_PORT), timeout=5):
        mqtt_available = True
except Exception as exc:
    mqtt_error = exc
print('mqtt_available', mqtt_available)
print('mqtt_error', type(mqtt_error).__name__, mqtt_error)

# Find candidate anomaly payloads
model = get_default_isolation_forest()
candidates = [
    [500.0, 10.0, 10.0, 10.0],
    [300.0, 20.0, 20.0, 20.0],
    [2000.0, 30.0, 20.0, 20.0],
    [10.0, 0.0, 0.0, 0.0],
    [50.0, 5.0, 5.0, 5.0],
]
for c in candidates:
    r = detect_anomaly(c, model)
    print('candidate', c, 'anomaly', r['anomaly_detected'], 'score', r['anomaly_score'])

anomaly_payload = next((c for c in candidates if detect_anomaly(c, model)['anomaly_detected']), None)
print('selected_anomaly_payload', anomaly_payload)
if anomaly_payload is None:
    raise RuntimeError('No anomaly payload found')

# Start the app to verify notification creation via real endpoint
uvicorn_cmd = [sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '127.0.0.1', '--port', '8001', '--log-level', 'warning']
proc = subprocess.Popen(uvicorn_cmd, cwd=backend_root, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

try:
    start = time.time()
    server_ready = False
    while time.time() - start < 30:
        try:
            r = requests.get('http://127.0.0.1:8001/health', timeout=3)
            if r.status_code == 200:
                server_ready = True
                break
        except Exception:
            pass
        time.sleep(1)

    print('server_ready', server_ready)
    if not server_ready:
        stderr = proc.stderr.read()
        print('server stderr\n', stderr)
        raise RuntimeError('Backend failed to start')

    base = 'http://127.0.0.1:8001'
    # Ensure device exists
    resp = requests.post(base + '/api/devices', json={'device_id': 'DEMO001', 'device_name': 'Smart Camera Demo', 'device_type': 'smart_camera'}, timeout=10)
    print('device_create', resp.status_code, resp.text)

    # Send anomaly analysis
    payload = {
        'packet_rate': anomaly_payload[0],
        'behavior_score': anomaly_payload[1],
        'network_score': anomaly_payload[2],
        'firmware_score': anomaly_payload[3],
    }
    analysis_resp = requests.post(base + '/api/devices/DEMO001/analyze', json=payload, timeout=10)
    print('analysis_status', analysis_resp.status_code)
    print('analysis_body', analysis_resp.json() if analysis_resp.headers.get('content-type','').startswith('application/json') else analysis_resp.text)

    # Check notification endpoints
    list_resp = requests.get(base + '/api/notifications', timeout=10)
    device_notif_resp = requests.get(base + '/api/notifications/DEMO001', timeout=10)
    print('notifications_all', list_resp.status_code, list_resp.text)
    print('notifications_device', device_notif_resp.status_code, device_notif_resp.text)

    # Verify MongoDB document
    client = MongoClient(settings.mongodb_uri, serverSelectionTimeoutMS=5000)
    client.admin.command('ping')
    db = client[settings.DATABASE_NAME]
    notification_doc = db['notifications'].find_one({'device_id': 'DEMO001'})
    print('mongo_notification_exists', notification_doc is not None)
    if notification_doc:
        print('mongo_notification_status', notification_doc.get('status'))
        print('mongo_notification_event_type', notification_doc.get('event_type'))

finally:
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except Exception:
        proc.kill()


MQTT host localhost
MQTT port 1883
MQTT topic iot/devices/data
MQTT auth configured False
mqtt_available False
mqtt_error ConnectionRefusedError [WinError 10061] No connection could be made because the target machine actively refused it
candidate [500.0, 10.0, 10.0, 10.0] anomaly True score -0.04790896545015266
candidate [300.0, 20.0, 20.0, 20.0] anomaly True score -0.04790896545015266
candidate [2000.0, 30.0, 20.0, 20.0] anomaly True score -0.04790896545015266
candidate [10.0, 0.0, 0.0, 0.0] anomaly True score -0.007506371346757468
candidate [50.0, 5.0, 5.0, 5.0] anomaly True score -0.04790896545015266
selected_anomaly_payload [500.0, 10.0, 10.0, 10.0]
server_ready True
device_create 409 {"success":false,"message":"Device with this device_id already exists","device_id":"DEMO001"}
analysis_status 200
analysis_body {'device_id': 'DEMO001', 'input_features': {'packet_rate': 500.0, 'behavior_score': 10.0, 'network_score': 10.0, 'firmware_score': 10.0}, 'anomaly_detected': True, 'anomaly_s

In [8]:
import subprocess
import sys
from pathlib import Path

backend_root = Path('c:/IOT_Project/backend')

# Ensure pytest is installed
res = subprocess.run([sys.executable, '-m', 'pip', 'show', 'pytest'], capture_output=True, text=True)
print('pytest installed:', bool(res.stdout.strip()))
if not res.stdout.strip():
    install = subprocess.run([sys.executable, '-m', 'pip', 'install', 'pytest'], capture_output=True, text=True)
    print('install stdout:', install.stdout)
    print('install stderr:', install.stderr)

# Run tests
res_q = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=backend_root, capture_output=True, text=True)
print('pytest -q returncode', res_q.returncode)
print(res_q.stdout)
print(res_q.stderr)

res_v = subprocess.run([sys.executable, '-m', 'pytest', '-v'], cwd=backend_root, capture_output=True, text=True)
print('pytest -v returncode', res_v.returncode)
print(res_v.stdout)
print(res_v.stderr)


pytest installed: True
pytest -q returncode 1
F..................                                                      [100%]
================================== FAILURES ===================================
_______________________ test_complete_analysis_pipeline _______________________
async def functions are not natively supported.
You need to install a suitable plugin for your async framework, for example:
  - anyio
  - pytest-asyncio
  - pytest-tornasync
  - pytest-trio
  - pytest-twisted
============================== warnings summary ===============================
app\main.py:86
  c:\IOT_Project\backend\app\main.py:86: DeprecationWarning: 
          on_event is deprecated, use lifespan event handlers instead.
  
          Read more about it in the
          [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
          
    @app.on_event("startup")

..\.venv\Lib\site-packages\fastapi\applications.py:4681
..\.venv\Lib\site-packages\fastapi\applications

In [7]:

# Install httpx2 if needed for FastAPI TestClient
httpx2_check = subprocess.run([sys.executable, '-m', 'pip', 'show', 'httpx2'], capture_output=True, text=True)
print('httpx2 installed:', bool(httpx2_check.stdout.strip()))
if not httpx2_check.stdout.strip():
    install_httpx2 = subprocess.run([sys.executable, '-m', 'pip', 'install', 'httpx2'], capture_output=True, text=True)
    print('httpx2 install stdout:', install_httpx2.stdout)
    print('httpx2 install stderr:', install_httpx2.stderr)


httpx2 installed: False
httpx2 install stdout: Collecting httpx2

   ---------------------------------------- 0/3 [truststore]
   ---------------------------------------- 0/3 [truststore]
   ---------------------------------------- 0/3 [truststore]
   ---------------------------------------- 0/3 [truststore]
   ------------- -------------------------- 1/3 [httpcore2]
   ------------- -------------------------- 1/3 [httpcore2]
   ------------- -------------------------- 1/3 [httpcore2]
   ------------- -------------------------- 1/3 [httpcore2]
   ------------- -------------------------- 1/3 [httpcore2]
   ------------- -------------------------- 1/3 [httpcore2]
   ------------- -------------------------- 1/3 [httpcore2]
   ------------- -------------------------- 1/3 [httpcore2]
   ------------- -------------------------- 1/3 [httpcore2]
   ------------- -------------------------- 1/3 [httpcore2]
   ------------- -------------------------- 1/3 [httpcore2]
   ------------- -------------

In [1]:
import time
from pathlib import Path
from dotenv import load_dotenv

import requests
from pymongo import MongoClient

from app.config import settings
from app.services.anomaly_service import get_default_isolation_forest, detect_anomaly

backend_root = Path('c:/IOT_Project/backend')
load_dotenv(dotenv_path=backend_root / '.env')

base = 'http://127.0.0.1:8000'
print('health', requests.get(base + '/health', timeout=10).status_code)
print('docs', requests.get(base + '/docs', timeout=10).status_code)

register = requests.post(base + '/api/devices', json={
    'device_id': 'DEMO001',
    'device_name': 'Smart Camera Demo',
    'device_type': 'smart_camera',
}, timeout=10)
print('register', register.status_code, register.text)

candidates = [
    {'packet_rate': 500, 'behavior_score': 10, 'network_score': 10, 'firmware_score': 10},
    {'packet_rate': 300, 'behavior_score': 20, 'network_score': 20, 'firmware_score': 20},
    {'packet_rate': 2000, 'behavior_score': 30, 'network_score': 20, 'firmware_score': 20},
    {'packet_rate': 10, 'behavior_score': 0, 'network_score': 0, 'firmware_score': 0},
    {'packet_rate': 50, 'behavior_score': 5, 'network_score': 5, 'firmware_score': 5},
]
model = get_default_isolation_forest()
for c in candidates:
    result = detect_anomaly([c['packet_rate'], c['behavior_score'], c['network_score'], c['firmware_score']], model)
    print('candidate', c, 'anomaly', result['anomaly_detected'], 'score', result['anomaly_score'])

anomaly_payload = next((c for c in candidates if detect_anomaly([c['packet_rate'], c['behavior_score'], c['network_score'], c['firmware_score']], model)['anomaly_detected']), None)
print('selected anomaly payload', anomaly_payload)

analysis = requests.post(base + '/api/devices/DEMO001/analyze', json=anomaly_payload, timeout=10)
print('analysis status', analysis.status_code)
print('analysis body', analysis.text)

# Wait briefly for notification creation.
time.sleep(1)

notif_all = requests.get(base + '/api/notifications', timeout=10)
notif_device = requests.get(base + '/api/notifications/DEMO001', timeout=10)
print('notifications all', notif_all.status_code, notif_all.text)
print('notifications device', notif_device.status_code, notif_device.text)

client = MongoClient(settings.mongodb_uri, serverSelectionTimeoutMS=5000)
client.admin.command('ping')
db = client[settings.DATABASE_NAME]
print('collection names', db.list_collection_names())
print('notification_doc_exists', db['notifications'].count_documents({'device_id': 'DEMO001'}) > 0)
print('notification_doc_sample', db['notifications'].find_one({'device_id': 'DEMO001'}))


health 200
docs 200
register 409 {"success":false,"message":"Device with this device_id already exists","device_id":"DEMO001"}
candidate {'packet_rate': 500, 'behavior_score': 10, 'network_score': 10, 'firmware_score': 10} anomaly True score -0.04790896545015266
candidate {'packet_rate': 300, 'behavior_score': 20, 'network_score': 20, 'firmware_score': 20} anomaly True score -0.04790896545015266
candidate {'packet_rate': 2000, 'behavior_score': 30, 'network_score': 20, 'firmware_score': 20} anomaly True score -0.04790896545015266
candidate {'packet_rate': 10, 'behavior_score': 0, 'network_score': 0, 'firmware_score': 0} anomaly True score -0.007506371346757468
candidate {'packet_rate': 50, 'behavior_score': 5, 'network_score': 5, 'firmware_score': 5} anomaly True score -0.04790896545015266
selected anomaly payload {'packet_rate': 500, 'behavior_score': 10, 'network_score': 10, 'firmware_score': 10}
analysis status 200
analysis body {"device_id":"DEMO001","input_features":{"packet_rate"

In [2]:
import subprocess
import sys
from pathlib import Path
from dotenv import load_dotenv
import socket
import time

import requests
from pymongo import MongoClient

from app.config import settings

backend_root = Path('c:/IOT_Project/backend')
load_dotenv(dotenv_path=backend_root / '.env')

print('=== MQTT CONFIG ===')
print('broker', settings.MQTT_BROKER)
print('port', settings.MQTT_PORT)
print('topic', settings.MQTT_TOPIC)
print('username configured', bool(settings.MQTT_USERNAME))
print('password configured', bool(settings.MQTT_PASSWORD))

mqtt_available = False
mqtt_error = None
try:
    with socket.create_connection((settings.MQTT_BROKER, settings.MQTT_PORT), timeout=5):
        mqtt_available = True
except Exception as exc:
    mqtt_error = exc
print('mqtt_available', mqtt_available)
print('mqtt_error', type(mqtt_error).__name__, mqtt_error)

print('\n=== FASTAPI RUNNING CHECK ===')
base = 'http://127.0.0.1:8000'
try:
    print('/health', requests.get(base + '/health', timeout=5).status_code)
except Exception as exc:
    print('/health error', exc)
try:
    print('/docs', requests.get(base + '/docs', timeout=5).status_code)
except Exception as exc:
    print('/docs error', exc)

print('\n=== PYTEST ===')
pytest_proc_q = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=backend_root, capture_output=True, text=True)
print('pytest -q returncode', pytest_proc_q.returncode)
print(pytest_proc_q.stdout)
print(pytest_proc_q.stderr)
pytest_proc_v = subprocess.run([sys.executable, '-m', 'pytest', '-v'], cwd=backend_root, capture_output=True, text=True)
print('pytest -v returncode', pytest_proc_v.returncode)
print(pytest_proc_v.stdout)
print(pytest_proc_v.stderr)


=== MQTT CONFIG ===
broker localhost
port 1883
topic iot/devices/data
username configured False
password configured False
mqtt_available False
mqtt_error ConnectionRefusedError [WinError 10061] No connection could be made because the target machine actively refused it

=== FASTAPI RUNNING CHECK ===
/health 200
/docs 200

=== PYTEST ===
pytest -q returncode 1
F..................                                                      [100%]
================================== FAILURES ===================================
_______________________ test_complete_analysis_pipeline _______________________
async def functions are not natively supported.
You need to install a suitable plugin for your async framework, for example:
  - anyio
  - pytest-asyncio
  - pytest-tornasync
  - pytest-trio
  - pytest-twisted
============================== warnings summary ===============================
app\main.py:86
  c:\IOT_Project\backend\app\main.py:86: DeprecationWarning: 
          on_event is deprecated

In [1]:
import socket
import subprocess
import sys
import time
from pathlib import Path

import requests
from dotenv import load_dotenv
from pymongo import MongoClient

from app.config import settings
from app.services.anomaly_service import get_default_isolation_forest, detect_anomaly

backend_root = Path('c:/IOT_Project/backend')
load_dotenv(dotenv_path=backend_root / '.env')

print('=== MQTT CONFIG ===')
print('broker', settings.MQTT_BROKER)
print('port', settings.MQTT_PORT)
print('topic', settings.MQTT_TOPIC)
print('username configured', bool(settings.MQTT_USERNAME))
print('password configured', bool(settings.MQTT_PASSWORD))

mqtt_available = False
mqtt_error = None
try:
    with socket.create_connection((settings.MQTT_BROKER, settings.MQTT_PORT), timeout=5):
        mqtt_available = True
except Exception as exc:
    mqtt_error = exc
print('mqtt_available', mqtt_available)
print('mqtt_error', type(mqtt_error).__name__, mqtt_error)

print('\n=== FASTAPI CHECK ===')
base = 'http://127.0.0.1:8000'
try:
    print('/health', requests.get(base + '/health', timeout=10).status_code)
except Exception as exc:
    print('/health error', exc)
try:
    print('/docs', requests.get(base + '/docs', timeout=10).status_code)
except Exception as exc:
    print('/docs error', exc)

# Ensure DEMO001 exists
print('\n=== DEVICE REGISTRATION ===')
reg = requests.post(base + '/api/devices', json={
    'device_id': 'DEMO001',
    'device_name': 'Smart Camera Demo',
    'device_type': 'smart_camera',
}, timeout=10)
print('register', reg.status_code, reg.text)

# Find a candidate anomaly payload that triggers the backend model
candidates = [
    {'packet_rate': 500, 'behavior_score': 10, 'network_score': 10, 'firmware_score': 10},
    {'packet_rate': 300, 'behavior_score': 20, 'network_score': 20, 'firmware_score': 20},
    {'packet_rate': 2000, 'behavior_score': 30, 'network_score': 20, 'firmware_score': 20},
    {'packet_rate': 10, 'behavior_score': 0, 'network_score': 0, 'firmware_score': 0},
    {'packet_rate': 50, 'behavior_score': 5, 'network_score': 5, 'firmware_score': 5},
]
model = get_default_isolation_forest()
for c in candidates:
    r = detect_anomaly([c['packet_rate'], c['behavior_score'], c['network_score'], c['firmware_score']], model)
    print('candidate', c, 'anomaly', r['anomaly_detected'], 'score', r['anomaly_score'])

anomaly_payload = next(
    (c for c in candidates if detect_anomaly([c['packet_rate'], c['behavior_score'], c['network_score'], c['firmware_score']], model)['anomaly_detected']),
    None,
)
print('selected anomaly payload', anomaly_payload)

print('\n=== ANALYSIS FOR NOTIFICATION ===')
an_resp = requests.post(base + '/api/devices/DEMO001/analyze', json=anomaly_payload, timeout=10)
print('analysis_status', an_resp.status_code)
print('analysis_body', an_resp.text)

# Wait for notification persistence.
time.sleep(1)

notif_all = requests.get(base + '/api/notifications', timeout=10)
notif_dev = requests.get(base + '/api/notifications/DEMO001', timeout=10)
print('\n=== NOTIFICATIONS ===')
print('notifications_all', notif_all.status_code, notif_all.text)
print('notifications_device', notif_dev.status_code, notif_dev.text)

print('\n=== MONGODB NOTIFICATION DOCUMENT ===')
client = MongoClient(settings.mongodb_uri, serverSelectionTimeoutMS=5000)
client.admin.command('ping')
db = client[settings.DATABASE_NAME]
print('notification_doc_exists', db['notifications'].count_documents({'device_id': 'DEMO001'}) > 0)
print('notification_doc_sample', db['notifications'].find_one({'device_id': 'DEMO001'}))

print('\n=== PYTEST SUITE ===')
pytest_proc_q = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=backend_root, capture_output=True, text=True)
print('pytest -q returncode', pytest_proc_q.returncode)
print(pytest_proc_q.stdout)
print(pytest_proc_q.stderr)
pytest_proc_v = subprocess.run([sys.executable, '-m', 'pytest', '-v'], cwd=backend_root, capture_output=True, text=True)
print('pytest -v returncode', pytest_proc_v.returncode)
print(pytest_proc_v.stdout)
print(pytest_proc_v.stderr)


=== MQTT CONFIG ===
broker localhost
port 1883
topic iot/devices/data
username configured False
password configured False
mqtt_available False
mqtt_error ConnectionRefusedError [WinError 10061] No connection could be made because the target machine actively refused it

=== FASTAPI CHECK ===
/health 200
/docs 200

=== DEVICE REGISTRATION ===
register 409 {"success":false,"message":"Device with this device_id already exists","device_id":"DEMO001"}
candidate {'packet_rate': 500, 'behavior_score': 10, 'network_score': 10, 'firmware_score': 10} anomaly True score -0.04790896545015266
candidate {'packet_rate': 300, 'behavior_score': 20, 'network_score': 20, 'firmware_score': 20} anomaly True score -0.04790896545015266
candidate {'packet_rate': 2000, 'behavior_score': 30, 'network_score': 20, 'firmware_score': 20} anomaly True score -0.04790896545015266
candidate {'packet_rate': 10, 'behavior_score': 0, 'network_score': 0, 'firmware_score': 0} anomaly True score -0.007506371346757468
candidat

In [ ]:
import subprocess

print('node version:', subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip())
print('npm version:', subprocess.run(['npm', '--version'], capture_output=True, text=True).stdout.strip())


In [ ]:
import subprocess

for tool in ['node', 'npm', 'npx']:
    try:
        result = subprocess.run([tool, '--version'], capture_output=True, text=True, timeout=10)
        print(f'{tool} version:', result.stdout.strip() or result.stderr.strip())
    except Exception as exc:
        print(f'{tool} check failed:', exc)


In [ ]:
import sys
import subprocess

print('platform', sys.platform)
for cmd in [['node', '--version'], ['npm', '--version'], ['npx', '--version'], ['where', 'node'], ['where', 'npm']]:
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        print('cmd', cmd, 'return', p.returncode)
        print('stdout', p.stdout.strip())
        print('stderr', p.stderr.strip())
    except Exception as exc:
        print('cmd', cmd, 'failed', type(exc).__name__, exc)


In [12]:
import subprocess

for cmd in [['node', '--version'], ['npm', '--version'], ['npx', '--version']]:
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        print(' '.join(cmd), '->', result.returncode)
        print('stdout:', result.stdout.strip())
        print('stderr:', result.stderr.strip())
    except Exception as exc:
        print(' '.join(cmd), 'failed:', type(exc).__name__, exc)


node --version -> 0
stdout: v24.18.0
stderr: 
npm --version failed: FileNotFoundError [WinError 2] The system cannot find the file specified
npx --version failed: FileNotFoundError [WinError 2] The system cannot find the file specified


In [5]:
import subprocess

for cmd in [['corepack', '--version'], ['node', '-p', 'process.execPath'], ['node', '-p', 'process.env.PATH']]:
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        print(' '.join(cmd), '->', result.returncode)
        print('stdout:', result.stdout.strip())
        print('stderr:', result.stderr.strip())
    except Exception as exc:
        print(' '.join(cmd), 'failed:', type(exc).__name__, exc)


corepack --version failed: FileNotFoundError [WinError 2] The system cannot find the file specified
node -p process.execPath -> 0
stdout: C:\Program Files\nodejs\node.exe
stderr: 
node -p process.env.PATH -> 0
stdout: c:\IOT_Project\.venv\Scripts;C:\Program Files\Common Files\Oracle\Java\javapath;C:\Python314\Scripts\;C:\Python314\;C:\WINDOWS\system32;C:\WINDOWS;C:\WINDOWS\System32\Wbem;C:\WINDOWS\System32\WindowsPowerShell\v1.0\;C:\WINDOWS\System32\OpenSSH\;C:\Program Files\Cloudflare\Cloudflare WARP\;C:\Program Files\nodejs\;C:\ProgramData\chocolatey\bin;C:\Program Files\Git\cmd;C:\Program Files\Amazon\AWSCLIV2\;C:\Program Files\GitHub CLI\;C:\Program Files\dotnet\;C:\Users\RHYTHAN S\AppData\Local\Programs\Python\Python311\Scripts\;C:\Users\RHYTHAN S\AppData\Local\Programs\Python\Python311\;C:\Program Files\MySQL\MySQL Shell 8.0\bin\;C:\Users\RHYTHAN S\AppData\Local\Microsoft\WindowsApps;C:\Users\RHYTHAN S\AppData\Local\Programs\Microsoft VS Code\bin;C:\Users\RHYTHAN S\AppData\Roamin

In [9]:
import subprocess

print('Running npm diagnostics from cmd.exe shell...')
for command in [
    ['cmd.exe', '/c', 'cd /d C:\\IOT_Project\\frontend && npm --version'],
    ['cmd.exe', '/c', 'cd /d C:\\IOT_Project\\frontend && npx --version'],
    ['cmd.exe', '/c', 'cd /d C:\\IOT_Project\\frontend && where npm'],
    ['cmd.exe', '/c', 'cd /d C:\\IOT_Project\\frontend && where npx'],
]:
    try:
        p = subprocess.run(command, capture_output=True, text=True, timeout=30)
        print('COMMAND:', ' '.join(command))
        print('returncode:', p.returncode)
        print('stdout:', p.stdout.strip())
        print('stderr:', p.stderr.strip())
    except Exception as exc:
        print('COMMAND ERROR:', exc)

print('\nAttempting npm install...')
try:
    p = subprocess.run(['cmd.exe', '/c', 'cd /d C:\\IOT_Project\\frontend && npm install'], capture_output=True, text=True, timeout=600)
    print('npm install returncode:', p.returncode)
    print('stdout:', p.stdout[:8000])
    print('stderr:', p.stderr[:8000])
except Exception as exc:
    print('npm install ERROR:', exc)


Running npm diagnostics from cmd.exe shell...
COMMAND: cmd.exe /c cd /d C:\IOT_Project\frontend && npm --version
returncode: 0
stdout: 11.16.0
stderr: 
COMMAND: cmd.exe /c cd /d C:\IOT_Project\frontend && npx --version
returncode: 0
stdout: 11.16.0
stderr: 
COMMAND: cmd.exe /c cd /d C:\IOT_Project\frontend && where npm
returncode: 0
stdout: C:\Program Files\nodejs\npm
C:\Program Files\nodejs\npm.cmd
stderr: 
COMMAND: cmd.exe /c cd /d C:\IOT_Project\frontend && where npx
returncode: 0
stdout: C:\Program Files\nodejs\npx
C:\Program Files\nodejs\npx.cmd
stderr: 

Attempting npm install...
npm install returncode: 0
stdout: 
up to date, audited 92 packages in 2s

13 packages are looking for funding
  run `npm fund` for details

4 vulnerabilities (3 moderate, 1 high)

To address issues that do not require attention, run:
  npm audit fix

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.

stderr: npm warn allow-scripts 1 package has

In [6]:
import os
import subprocess
from pathlib import Path

commands = [
    ['node', '--version'],
    ['npm', '--version'],
    ['npx', '--version'],
]

for cmd in commands:
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        print('COMMAND:', ' '.join(cmd))
        print('returncode:', p.returncode)
        print('stdout:', p.stdout.strip())
        print('stderr:', p.stderr.strip())
    except Exception as exc:
        print('COMMAND:', ' '.join(cmd), 'ERROR:', type(exc).__name__, exc)

print('\nWHERE LOOKUP')
for item in ['node', 'npm', 'npx']:
    try:
        p = subprocess.run(['where', item], capture_output=True, text=True, timeout=15)
        print(item, 'returncode:', p.returncode)
        print(item, 'stdout:')
        print(p.stdout.strip())
        print(item, 'stderr:')
        print('---')
    except Exception as exc:
        print(item, 'ERROR:', type(exc).__name__, exc)

node_dir = Path(r'C:\Program Files\nodejs')
print('\nNODE INSTALLATION DIRECTORY:', node_dir)
print('exists:', node_dir.exists())
if node_dir.exists():
    for path in ['npm', 'npm.cmd', 'npm.ps1', 'npx', 'npx.cmd', 'npx.ps1', 'node.exe']:
        p = node_dir / path
        print(p.name, 'exists:', p.exists())

print('\nPATH CONTENT')
print(os.environ.get('PATH'))


COMMAND: node --version
returncode: 0
stdout: v24.18.0
stderr: 
COMMAND: npm --version ERROR: FileNotFoundError [WinError 2] The system cannot find the file specified
COMMAND: npx --version ERROR: FileNotFoundError [WinError 2] The system cannot find the file specified

WHERE LOOKUP
node returncode: 0
node stdout:
C:\Program Files\nodejs\node.exe
node stderr:
---
npm returncode: 0
npm stdout:
C:\Program Files\nodejs\npm
C:\Program Files\nodejs\npm.cmd
npm stderr:
---
npx returncode: 0
npx stdout:
C:\Program Files\nodejs\npx
C:\Program Files\nodejs\npx.cmd
npx stderr:
---

NODE INSTALLATION DIRECTORY: C:\Program Files\nodejs
exists: True
npm exists: True
npm.cmd exists: True
npm.ps1 exists: True
npx exists: True
npx.cmd exists: True
npx.ps1 exists: True
node.exe exists: True

PATH CONTENT
c:\IOT_Project\.venv\Scripts;C:\Program Files\Common Files\Oracle\Java\javapath;C:\Python314\Scripts\;C:\Python314\;C:\WINDOWS\system32;C:\WINDOWS;C:\WINDOWS\System32\Wbem;C:\WINDOWS\System32\WindowsPo

In [10]:
print('\nRunning frontend production build...')
try:
    p = subprocess.run(['cmd.exe', '/c', 'cd /d C:\\IOT_Project\\frontend && npm run build'], capture_output=True, text=True, timeout=300)
    print('build returncode:', p.returncode)
    print('build stdout:', p.stdout[:8000])
    print('build stderr:', p.stderr[:8000])
except Exception as exc:
    print('build ERROR:', type(exc).__name__, exc)



Running frontend production build...
build returncode: 0
build stdout: 
> iot-trust-drift-frontend@0.1.0 build
> vite build

vite v5.4.21 building for production...
transforming...
âœ“ 98 modules transformed.
rendering chunks...
computing gzip size...
dist/index.html                   0.41 kB â”‚ gzip:  0.28 kB
dist/assets/index-C05FWgcI.css    4.02 kB â”‚ gzip:  1.42 kB
dist/assets/index-5rHwLbhV.js   227.56 kB â”‚ gzip: 75.07 kB
âœ“ built in 3.12s

build stderr: 


In [11]:
print('\nStarting frontend dev server for smoke test...')

import time
from threading import Thread

server_output = []

try:
    proc = subprocess.Popen(['cmd.exe', '/c', 'cd /d C:\\IOT_Project\\frontend && npm run dev -- --host 127.0.0.1 --port 4173'],
                            stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT,
                            text=True)

    start_time = time.time()
    while time.time() - start_time < 20:
        line = proc.stdout.readline()
        if not line:
            break
        server_output.append(line)
        print(line, end='')
        if 'Local:' in line or 'running at' in line.lower():
            break
    else:
        print('Dev server did not report startup in time')
except Exception as exc:
    print('Dev server start failed:', exc)
finally:
    if proc and proc.poll() is None:
        proc.terminate()
        proc.wait(timeout=10)
        print('Dev server terminated')



Starting frontend dev server for smoke test...
npm error Missing script: "dev"
npm error
npm error To see a list of scripts, run:
npm error   npm run
npm error A complete log of this run can be found in: C:\Users\RHYTHAN S\AppData\Local\npm-cache\_logs\2026-08-12T13_43_09_553Z-debug-0.log
Dev server terminated


In [7]:

print('\nCMD.EXE SHELL CHECK')
for command in [['cmd.exe', '/c', 'npm --version'], ['cmd.exe', '/c', 'npx --version'], ['cmd.exe', '/c', 'where npm'], ['cmd.exe', '/c', 'where npx']]:
    try:
        p = subprocess.run(command, capture_output=True, text=True, timeout=15)
        print('COMMAND:', ' '.join(command))
        print('returncode:', p.returncode)
        print('stdout:', p.stdout.strip())
        print('stderr:', p.stderr.strip())
    except Exception as exc:
        print('COMMAND:', ' '.join(command), 'ERROR:', type(exc).__name__, exc)



CMD.EXE SHELL CHECK
COMMAND: cmd.exe /c npm --version
returncode: 0
stdout: 11.16.0
stderr: 
COMMAND: cmd.exe /c npx --version
returncode: 0
stdout: 11.16.0
stderr: 
COMMAND: cmd.exe /c where npm
returncode: 0
stdout: C:\Program Files\nodejs\npm
C:\Program Files\nodejs\npm.cmd
stderr: 
COMMAND: cmd.exe /c where npx
returncode: 0
stdout: C:\Program Files\nodejs\npx
C:\Program Files\nodejs\npx.cmd
stderr: 


In [ ]:
import subprocess
import time
import requests
from pathlib import Path

frontend_dir = Path(r'C:\IOT_Project\frontend')
commands = ['cmd.exe', '/c', 'cd /d', str(frontend_dir), '&&', 'npm', 'run', 'dev', '--', '--host', '127.0.0.1', '--port', '4173']
proc = subprocess.Popen(commands, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('Started frontend server process, pid=', proc.pid)

frontend_url = 'http://127.0.0.1:4173'
backend_url = 'http://127.0.0.1:8000'
started = False
start_time = time.time()

try:
    while time.time() - start_time < 30:
        line = proc.stdout.readline()
        if line:
            print(line, end='')
            if 'Local:' in line or 'running at' in line.lower() or frontend_url in line:
                started = True
                break
        else:
            time.sleep(0.2)

    if not started:
        print('Frontend dev server did not report startup in time')
    else:
        print('Frontend server reported startup, checking page...')
        success = False
        for _ in range(12):
            try:
                response = requests.get(frontend_url, timeout=5)
                print('Frontend page status:', response.status_code)
                print('HTML head:', response.text[:420])
                success = response.status_code == 200
                break
            except Exception as exc:
                print('Waiting for frontend HTTP response...', exc)
                time.sleep(1)
        if not success:
            print('Frontend HTTP page did not respond with status 200')

        print('\nChecking backend health...')
        try:
            health = requests.get(f'{backend_url}/health', timeout=5)
            print('/health status:', health.status_code, health.text)
        except Exception as exc:
            print('/health request failed:', exc)

        print('\nChecking CORS on backend /health (Origin header)...')
        try:
            response = requests.get(f'{backend_url}/health', headers={'Origin': 'http://127.0.0.1:4173'}, timeout=5)
            print('CORS status:', response.status_code)
            print('Access-Control-Allow-Origin:', response.headers.get('access-control-allow-origin'))
        except Exception as exc:
            print('CORS check failed:', exc)

        print('\nChecking dashboard and devices APIs...')
        for path in ['/api/dashboard/summary', '/api/dashboard/devices', '/api/devices']:
            try:
                r = requests.get(f'{backend_url}{path}', timeout=5)
                print(path, r.status_code, r.text[:320])
            except Exception as exc:
                print(path, 'failed:', exc)

        print('\nChecking DEMO001 details and notifications...')
        for path in ['/api/devices/DEMO001', '/api/notifications', '/api/notifications/DEMO001']:
            try:
                r = requests.get(f'{backend_url}{path}', timeout=5)
                print(path, r.status_code, r.text[:320])
            except Exception as exc:
                print(path, 'failed:', exc)

        print('\nSubmitting anomaly analysis to generate backend result...')
        analyze_payload = {
            'packet_rate': 500,
            'behavior_score': 10,
            'network_score': 10,
            'firmware_score': 10,
        }
        try:
            r = requests.post(f'{backend_url}/api/devices/DEMO001/analyze', json=analyze_payload, timeout=10)
            print('/api/devices/DEMO001/analyze', r.status_code, r.text)
        except Exception as exc:
            print('Analyze request failed:', exc)

finally:
    if proc.poll() is None:
        proc.terminate()
        proc.wait(timeout=10)
        print('Dev server terminated')


Started frontend server process, pid= 22760

> iot-trust-drift-frontend@0.1.0 dev
> vite --host 127.0.0.1 --port 4173


  VITE v5.4.21  ready in 617 ms

  âžœ  Local:   http://127.0.0.1:4173/
